# NB05: Build Reference Metabolic Models

Build genome-scale metabolic models for reference organisms using modelseedpy's
template-based reconstruction pipeline. Genomes are obtained from BV-BRC and
models are built using ModelSEED templates (v7).

**Organisms:**
- **E. coli K-12 MG1655** (511145.12) — Gram-negative reference
- **B. subtilis 168** (224308.43) — Gram-positive reference

**Pipeline:** BV-BRC genome → MSGenome (RAST ontology) → MSBuilder → ATP correction → Gapfilling → FBA verification

In [ ]:
import json
import sys
from pathlib import Path

import cobra
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from modelseedpy.core.mstemplate import MSTemplateBuilder
from modelseedpy.core.msgenome import MSGenome, MSFeature
from modelseedpy import MSBuilder, MSMedia, MSATPCorrection, MSGapfill
from modelseedpy.core.msatpcorrection import load_default_medias
from modelseedpy.core.msmodel import get_reaction_constraints_from_direction

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
FIGURES_DIR = PROJECT_ROOT / 'figures'

sys.path.insert(0, '/global_share/KBaseUtilities/KBUtilLib/src')
from kbutillib.bvbrc_utils import BVBRCUtils
BVBRCUtils.save = lambda self, name, obj: None

## 1. Load ModelSEED Templates

In [ ]:
TEMPLATE_DIR = Path('/tmp/ModelSEEDTemplates/templates/v7.0')

templates = {}
for tname, fname in [
    ('GramNeg', 'GramNegModelTemplateV7.json'),
    ('GramPos', 'GramPosModelTemplateV7.json'),
    ('Core', 'Core-V6.json'),
]:
    with open(TEMPLATE_DIR / fname) as f:
        data = json.load(f)
    templates[tname] = MSTemplateBuilder.from_dict(data, None).build()
    t = templates[tname]
    print(f'{tname}: {len(t.reactions)} reactions, {len(t.compounds)} compounds, {len(t.biomasses)} biomasses')

## 2. Define Reference Organisms

In [ ]:
reference_organisms = {
    'ecoli': {
        'name': 'Escherichia coli K-12 MG1655',
        'bvbrc_id': '511145.12',
        'template': 'GramNeg',
    },
    'bsubtilis': {
        'name': 'Bacillus subtilis 168',
        'bvbrc_id': '224308.43',
        'template': 'GramPos',
    },
}

## 3. Download Genomes from BV-BRC

In [ ]:
bvbrc = BVBRCUtils()

def build_msgenome(genome_dict):
    """Convert BV-BRC genome dict to MSGenome with RAST ontology terms."""
    genome = MSGenome()
    genome.id = genome_dict['id']
    genome.scientific_name = genome_dict['scientific_name']
    genome.domain = genome_dict.get('domain', 'Bacteria')
    features = []
    func_count = 0
    for feat_dict in genome_dict['features']:
        seq = feat_dict.get('protein_translation', '')
        if not seq:
            continue
        funcs = feat_dict.get('functions', [])
        desc = ' / '.join(funcs) if funcs else ''
        aliases = feat_dict.get('aliases', [])
        f = MSFeature(feat_dict['id'], seq, description=desc, aliases=aliases)
        f.ontology_terms = {}
        roles = []
        for func_str in funcs:
            for role in func_str.split(' / '):
                role = role.strip()
                if role and role != 'hypothetical protein':
                    roles.append(role)
        if roles:
            f.ontology_terms['RAST'] = roles
            func_count += 1
        features.append(f)
    genome.add_features(features)
    return genome, func_count

genomes = {}
for org_id, org_info in reference_organisms.items():
    cache_path = DATA_DIR / f'genome_{org_id}.json'
    if cache_path.exists():
        print(f'Loading cached genome for {org_info["name"]}...')
        with open(cache_path) as f:
            genome_dict = json.load(f)
    else:
        print(f'Downloading {org_info["name"]} from BV-BRC...')
        genome_dict = bvbrc.build_kbase_genome_from_api(org_info['bvbrc_id'])
        with open(cache_path, 'w') as f:
            json.dump(genome_dict, f)
    
    genome, func_count = build_msgenome(genome_dict)
    genomes[org_id] = genome
    print(f'  {org_id}: {len(genome.features)} features, {func_count} with functional annotations')

## 4. Build Models

Pipeline: MSBuilder (template-based) → ATP correction → Gapfilling on complete media

In [ ]:
atp_medias = load_default_medias()
core_template = templates['Core']
models = {}
build_results = []

for org_id, org_info in reference_organisms.items():
    print(f'\n=== Building model for {org_info["name"]} ===')
    genome = genomes[org_id]
    template = templates[org_info['template']]
    
    # Build base model
    builder = MSBuilder(genome, template=template, template_core=core_template)
    base_model = cobra.Model(id_or_model=genome.id, name=genome.scientific_name)
    model = builder.build_base_model(base_model, index='0', annotate_with_rast=False)
    MSBuilder.add_atpm(model)
    n_base = len(model.reactions)
    n_genes = len(model.genes)
    print(f'  Base model: {n_base} reactions, {n_genes} genes')
    
    # ATP correction
    atp_correction = MSATPCorrection(
        model, core_template, atp_medias,
        compartment='c0', atp_hydrolysis_id='ATPM_c0',
        load_default_medias=False,
    )
    atp_correction.evaluate_growth_media()
    atp_correction.determine_growth_media()
    atp_correction.apply_growth_media_gapfilling()
    atp_correction.expand_model_to_genome_scale()
    tests = atp_correction.build_tests()
    n_atp = len(model.reactions) - n_base
    print(f'  ATP correction: +{n_atp} reactions, {len(tests)} test conditions')
    
    # Gapfilling
    complete_media_dict = {}
    for rxn in model.reactions:
        if rxn.id.startswith('EX_'):
            cpd_id = rxn.id.replace('EX_', '').replace('_e0', '')
            complete_media_dict[cpd_id] = 1000
    for cpd in ['cpd00001', 'cpd00067', 'cpd00009', 'cpd00027', 'cpd00007']:
        complete_media_dict[cpd] = 1000
    complete_media = MSMedia.from_dict(complete_media_dict)
    complete_media.id = 'Complete'
    complete_media.name = 'Complete'
    
    gapfill = MSGapfill(
        model, default_gapfill_templates=[template],
        test_conditions=tests, default_target='bio1',
    )
    gapfill_res = gapfill.run_gapfilling(media=complete_media)
    
    gap_sol = {}
    for rxn_id, d in gapfill_res.get('new', {}).items():
        if rxn_id[:-1] in template.reactions:
            gap_sol[rxn_id[:-1]] = get_reaction_constraints_from_direction(d)
    
    model_gf = model.copy()
    MSBuilder.integrate_gapfill_solution(template, model_gf, gap_sol)
    print(f'  Gapfilling: +{len(gap_sol)} reactions')
    
    # Verify growth
    sol = model_gf.optimize()
    print(f'  Final: {len(model_gf.reactions)} reactions, {len(model_gf.genes)} genes, growth={sol.objective_value:.4f}')
    
    models[org_id] = model_gf
    ms_rxns = [r for r in model_gf.reactions if r.id.startswith('rxn')]
    build_results.append({
        'organism': org_id,
        'name': org_info['name'],
        'template': org_info['template'],
        'genes': n_genes,
        'base_reactions': n_base,
        'atp_added': n_atp,
        'gapfill_added': len(gap_sol),
        'total_reactions': len(model_gf.reactions),
        'modelseed_reactions': len(ms_rxns),
        'growth': sol.objective_value,
    })

df_build = pd.DataFrame(build_results)
print('\n=== Build Summary ===')
print(df_build.to_string(index=False))

## 5. Map reactions to deltaG predictions

In [ ]:
with open(DATA_DIR / 'dg_predictions_marvin.json') as f:
    marvin_dg = json.load(f)
with open(DATA_DIR / 'dg_predictions_opam2.json') as f:
    opam2_dg = json.load(f)

print(f'Marvin predictions: {len(marvin_dg)} reactions')
print(f'OPAM2 predictions: {len(opam2_dg)} reactions')

for org_id, model in models.items():
    ms_rxns = [r.id.replace('_c0', '') for r in model.reactions if r.id.startswith('rxn')]
    in_marvin = sum(1 for r in ms_rxns if r in marvin_dg)
    in_opam2 = sum(1 for r in ms_rxns if r in opam2_dg)
    in_both = sum(1 for r in ms_rxns if r in marvin_dg and r in opam2_dg)
    print(f'\n{org_id}: {len(ms_rxns)} ModelSEED reactions')
    print(f'  With Marvin dG: {in_marvin} ({100*in_marvin/len(ms_rxns):.1f}%)')
    print(f'  With OPAM2 dG: {in_opam2} ({100*in_opam2/len(ms_rxns):.1f}%)')
    print(f'  With both: {in_both} ({100*in_both/len(ms_rxns):.1f}%)')

## 6. Save models and summary

In [ ]:
for org_id, model in models.items():
    model_path = DATA_DIR / f'model_{org_id}.json'
    cobra.io.save_json_model(model, str(model_path))
    print(f'Saved {org_id} model to {model_path}')

df_build.to_csv(DATA_DIR / 'model_build_summary.tsv', sep='\t', index=False)
print(f'Saved build summary to data/model_build_summary.tsv')